# Notebook 1: Data Exploration
### DL Capstone — Hair Change Tracking via U-Net Segmentation
**Ramdeobaba University | Dept. of Electronics Engineering | Semester V**

This notebook explores the **CelebAMask-HQ** dataset used for training the hair segmentation model.

---

In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

# Paths
ROOT       = Path(r'C:\Users\Aliasgar\Desktop\DL Capstone\DL_Capstone_HairSegmentation')
IMG_DIR    = ROOT / 'data' / 'processed' / 'images'
MASK_DIR   = ROOT / 'data' / 'processed' / 'masks'
SPLITS_DIR = ROOT / 'data' / 'splits'

print('Project root:', ROOT)
print('Images dir  :', IMG_DIR)
print('Masks dir   :', MASK_DIR)

## 1. Dataset Statistics

In [ ]:
all_imgs  = sorted(IMG_DIR.glob('*.png'))
all_masks = sorted(MASK_DIR.glob('*.png'))

# Load split sizes
train_ids = (SPLITS_DIR / 'train.txt').read_text().strip().split('\n')
val_ids   = (SPLITS_DIR / 'val.txt').read_text().strip().split('\n')
test_ids  = (SPLITS_DIR / 'test.txt').read_text().strip().split('\n')

print('=' * 50)
print(f'  Total processed images : {len(all_imgs):,}')
print(f'  Total masks            : {len(all_masks):,}')
print(f'  Train split            : {len(train_ids):,} ({len(train_ids)/len(all_imgs)*100:.1f}%)')
print(f'  Val split              : {len(val_ids):,} ({len(val_ids)/len(all_imgs)*100:.1f}%)')
print(f'  Test split             : {len(test_ids):,} ({len(test_ids)/len(all_imgs)*100:.1f}%)')
print(f'  Image size             : 256 x 256 (RGB PNG)')
print(f'  Mask type              : Binary (0=background, 255=hair)')
print('=' * 50)

## 2. Sample Images with Hair Mask Overlays

In [ ]:
random.seed(42)
sample_ids = random.sample(all_imgs, 8)

fig, axes = plt.subplots(8, 3, figsize=(12, 32))
fig.suptitle('Sample Images | Binary Hair Masks | Overlays', fontsize=14, fontweight='bold', y=1.01)

for i, img_path in enumerate(sample_ids):
    mask_path = MASK_DIR / img_path.name
    img  = np.array(Image.open(img_path).convert('RGB'))
    mask = np.array(Image.open(mask_path).convert('L'))

    # Overlay: red tint where hair
    overlay = img.copy()
    hair_px  = mask > 128
    overlay[hair_px] = (overlay[hair_px] * 0.5 + np.array([220, 50, 50]) * 0.5).astype(np.uint8)

    coverage = hair_px.sum() / hair_px.size * 100

    axes[i, 0].imshow(img);                  axes[i, 0].set_title(f'Input [{img_path.stem}]', fontsize=8)
    axes[i, 1].imshow(mask, cmap='gray');    axes[i, 1].set_title(f'Hair Mask ({coverage:.1f}% hair)', fontsize=8)
    axes[i, 2].imshow(overlay);              axes[i, 2].set_title('Overlay', fontsize=8)
    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig(str(ROOT / 'outputs' / 'plots' / 'sample_data_grid.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved -> outputs/plots/sample_data_grid.png')

## 3. Hair Coverage Distribution

In [ ]:
print('Computing hair coverage for all images (this takes ~1 minute)...')

# Sample 1000 random masks for speed
random.seed(42)
sampled = random.sample(all_masks, min(1000, len(all_masks)))

coverages = []
for mp in sampled:
    mask = np.array(Image.open(mp).convert('L'))
    coverages.append((mask > 128).sum() / mask.size * 100)

coverages = np.array(coverages)
print(f'\nHair Coverage Statistics (n={len(coverages):,} images):')
print(f'  Mean    : {coverages.mean():.2f}%')
print(f'  Std dev : {coverages.std():.2f}%')
print(f'  Min     : {coverages.min():.2f}%')
print(f'  Max     : {coverages.max():.2f}%')
print(f'  Median  : {np.median(coverages):.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(coverages, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(coverages.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {coverages.mean():.1f}%')
axes[0].axvline(np.median(coverages), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {np.median(coverages):.1f}%')
axes[0].set_xlabel('Hair Coverage (%)')
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Hair Coverage Distribution')
axes[0].legend()

# Class balance pie
avg_hair = coverages.mean()
avg_bg   = 100 - avg_hair
axes[1].pie([avg_hair, avg_bg], labels=['Hair pixels', 'Background pixels'],
            autopct='%1.1f%%', colors=['#e05050', '#5090d0'], startangle=90)
axes[1].set_title('Average Pixel-Level Class Balance')

plt.suptitle('CelebAMask-HQ — Hair Segmentation Dataset Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'outputs' / 'plots' / 'coverage_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved -> outputs/plots/coverage_distribution.png')

## 4. Dataset Summary Table

| Property | Value |
|---|---|
| Dataset | CelebAMask-HQ |
| Total images with hair masks | 29,300 |
| Skipped (no hair / bald) | 700 |
| Image resolution | 256 × 256 px (resized from 512×512) |
| Mask format | Binary PNG (0 = background, 255 = hair) |
| Train / Val / Test | 23,440 / 2,930 / 2,930 (80/10/10) |
| Augmentations | HFlip, Brightness/Contrast, GaussBlur, Affine |
| Normalization | ImageNet mean/std |